# Lezione 11 — Regressione logistica e modelli lineari generalizzati

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/11_regresion_logistica_glm_IT.md`

## Cosa vedrai in questa lezione

- Perché la regressione lineare non basta per un risultato 0/1, e il modello **logistico**: probabilità → odds → log-odds e la funzione sigmoide.
- Adattare con `statsmodels` (`smf.logit` e `sm.GLM` con famiglia Binomiale) e **leggere l'output**.
- I coefficienti sono **log-odds**; `exp(β)` è l'**odds ratio**. Odds ratio con intervallo, odds ratio rispetto al rischio relativo e variabili categoriali (dummy).
- Probabilità previste ed **effetto marginale medio**: la variazione di probabilità dipende dal punto di partenza.
- La **separazione perfetta** (un avvertimento) e la famiglia dei **GLM**: lineare, logistica e Poisson (quest'ultima solo come idea).

Tre esperimenti guidati: la retta rispetto alla logistica nelle previsioni, odds ratio rispetto a probabilità e l'effetto marginale.

## Prerequisiti

- Lezione 10 (regressione lineare e lettura di un risultato di `statsmodels`), lezioni 6 e 7 (errore standard, intervalli, p-value).
- Librerie: `numpy`, `pandas`, `matplotlib`, `statsmodels` (`pip install numpy pandas matplotlib statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. I dati di abbandono dei clienti (*churn*) sono sintetici: un *esempio illustrativo* generato con coefficienti veri noti, per poter confrontare la stima con la verità. La multicollinearità, la selezione delle variabili e la validazione si vedono nella lezione 12.

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme** e definiamo il generatore di clienti sintetici con i suoi **coefficienti veri** (`VERDAD`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coefficienti VERI con cui generiamo i dati (li conosciamo solo perché sono sintetici)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clienti sintetici (esempio illustrativo) con abbandono secondo VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # la useremo nella lezione 12
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

print("Librerie e funzioni caricate correttamente.")

---
## 1. I dati: abbandono dei clienti (*churn*)

Generiamo 2 000 clienti. La risposta `abandona` vale 1 se il cliente disdice. Le esplicative sono l'anzianità (mesi), la spesa mensile (€), i ticket di supporto e il piano annuale (0/1). Il dataset include anche `datos_gb`, che non useremo fino alla lezione 12.

In [ ]:
df = generar_clientes(2000, rng)

display(df.head())
print("Tasso di abbandono:", round(df["abandona"].mean() * 100, 1), "%")
display(df.describe().round(2))

---
## 2. Perché non basta la regressione lineare?

Adattiamo una regressione lineare (lezione 10) con `abandona` come risposta e guardiamo le previsioni.

In [ ]:
ols = smf.ols('abandona ~ antiguedad + gasto + tickets + plan_anual', data=df).fit()
pred_ols = ols.predict(df)
print("Previsione minima:", round(pred_ols.min(), 3), " massima:", round(pred_ols.max(), 3))
print("% di clienti con 'probabilità' fuori da [0, 1]:", round(((pred_ols < 0) | (pred_ols > 1)).mean() * 100, 1))

plt.hist(pred_ols, bins=40, edgecolor="white")
plt.axvline(0, color="red", linestyle="--")
plt.title("Previsioni della regressione lineare su un risultato 0/1")
plt.xlabel("'probabilità' prevista")
plt.show()

**Cosa osservi?** Ci sono previsioni negative: una "probabilità" minore di 0 non ha senso. Inoltre la risposta vale solo 0 o 1, quindi i residui non possono essere normali.

### Esperimento guidato 1: la retta e la logistica quando si estrapola

Ripetiamo lo studio **200 volte** con campioni di 500 clienti e adattiamo le due versioni. Guardiamo la previsione per due profili estremi: un cliente a rischio (6 mesi, 50 €, **8 ticket**, senza piano annuale) e uno molto stabile (72 mesi, 20 €, 0 ticket, con piano annuale). **Prima di eseguire, prevedi:** in quanti campioni la retta darà una "probabilità" fuori da [0, 1]? E la logistica?

In [ ]:
perfiles_ext = pd.DataFrame({"antiguedad": [6, 72], "gasto": [50, 20], "tickets": [8, 0], "plan_anual": [0, 1]},
                            index=["rischio", "stabile"])
rng_e1 = np.random.default_rng(123)
prev_ols, prev_logit = [], []
for _ in range(200):
    d = generar_clientes(500, rng_e1)
    prev_ols.append(smf.ols('abandona ~ antiguedad + gasto + tickets + plan_anual', data=d).fit().predict(perfiles_ext).values)
    prev_logit.append(smf.logit('abandona ~ antiguedad + gasto + tickets + plan_anual', data=d).fit(disp=0).predict(perfiles_ext).values)
prev_ols, prev_logit = np.array(prev_ols), np.array(prev_logit)

resumen = pd.DataFrame({
    "retta: % fuori da [0, 1]": ((prev_ols < 0) | (prev_ols > 1)).mean(axis=0) * 100,
    "retta: media": prev_ols.mean(axis=0),
    "logistica: minimo": prev_logit.min(axis=0),
    "logistica: massimo": prev_logit.max(axis=0)}, index=perfiles_ext.index)
display(resumen.round(3))

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for j, ax in enumerate(ejes):
    ax.hist(prev_ols[:, j], bins=25, alpha=0.6, label="retta")
    ax.hist(prev_logit[:, j], bins=25, alpha=0.6, label="logistica")
    ax.axvline(0, color="red", linestyle="--"); ax.axvline(1, color="red", linestyle="--")
    ax.set_title(perfiles_ext.index[j]); ax.legend()
plt.show()

**Cosa osservi?** La retta esce da [0, 1] in molti campioni (nel profilo stabile, in tutti): il problema non è un caso raro, è strutturale. La logistica resta sempre dentro, e nel profilo a rischio si avvicina a 1 senza superarlo. Con la logistica, il "tetto" e il "pavimento" sono incorporati.

---
## 3. Probabilità, odds e log-odds

`odds = p / (1 − p)` e `log-odds = ln(odds)`. L'inversa del logit è la **sigmoide**: `p = 1 / (1 + e^(−η))`.

In [ ]:
def a_odds(p):     return p / (1 - p)
def a_logodds(p):  return np.log(p / (1 - p))
def sigmoide(eta): return 1 / (1 + np.exp(-eta))

ps = np.array([0.10, 0.20, 0.50, 0.80, 0.90])
tabla = pd.DataFrame({"p": ps, "odds": a_odds(ps), "log-odds": a_logodds(ps)})
display(tabla.round(3))
print("Comprobazione: sigmoide(log-odds) restituisce p →", np.allclose(sigmoide(a_logodds(ps)), ps))

eta = np.linspace(-6, 6, 200)
plt.plot(eta, sigmoide(eta))
plt.axhline(0.5, color="gray", linestyle=":")
plt.title("La sigmoide: da log-odds a probabilità")
plt.xlabel("log-odds (η)")
plt.ylabel("p")
plt.show()

### Leggere un odds ratio: exp(0,69) ≈ 2

Un coefficiente β = 0,69 ha un odds ratio di `exp(0,69) ≈ 2`: gli **odds raddoppiano**. La probabilità raddoppia? Lo verifichiamo partendo da due probabilità diverse.

In [ ]:
beta = 0.69
or_ = np.exp(beta)
print("Odds ratio = exp(0,69) =", round(or_, 3))

for p0 in (0.20, 0.50):
    p1 = sigmoide(a_logodds(p0) + beta)          # sommare β ai log-odds = moltiplicare gli odds per l'OR
    print(f"Parte da p = {p0:.2f}  (odds {a_odds(p0):.2f})  →  p = {p1:.3f}  (odds {a_odds(p1):.2f})   rapporto tra probabilità: {p1 / p0:.2f}")

**Cosa osservi?** Gli odds raddoppiano sempre (da 0,25 a 0,50; da 1 a 2), ma la probabilità passa da 0,20 a 0,333 (×1,67) e da 0,50 a 0,667 (×1,33). L'odds ratio **non è** il rapporto tra probabilità (rischio relativo).

---
## 4. Massima verosimiglianza in piccolo

Tre clienti con risultati (1, 0, 1) e probabilità previste (0,8; 0,3; 0,6). La verosimiglianza è il prodotto delle probabilità che il modello assegnava a ciò che **è realmente successo**.

In [ ]:
y3 = np.array([1, 0, 1])
p3 = np.array([0.8, 0.3, 0.6])
prob_observada = np.where(y3 == 1, p3, 1 - p3)
verosimilitud = prob_observada.prod()
print("Probabilità di ciò che si è osservato, cliente per cliente:", prob_observada)
print("Verosimiglianza:", round(verosimilitud, 3), " Log-verosimiglianza:", round(np.log(verosimilitud), 3))

---
## 5. Adattare il modello e leggere l'output

In [ ]:
modelo = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=df).fit(disp=0)
print(modelo.summary().tables[1])     # tabella dei coefficienti (l'intestazione contiene data e ora, per questo la stampiamo a parte)
print("Osservazioni:", int(modelo.nobs), " convergenza:", modelo.mle_retvals["converged"])

Confrontiamo la stima con i coefficienti **veri** (che conosciamo solo perché i dati sono sintetici) e verifichiamo se ognuno cade dentro l'intervallo di confidenza al 95 %.

In [ ]:
ic = modelo.conf_int()
comparacion = pd.DataFrame({
    "vero": [VERDAD[v] for v in modelo.params.index],
    "stimato": modelo.params,
    "IC 2.5%": ic[0], "IC 97.5%": ic[1]})
comparacion["vero dentro l'IC?"] = (comparacion["IC 2.5%"] <= comparacion["vero"]) & (comparacion["vero"] <= comparacion["IC 97.5%"])
display(comparacion.round(4))
print("Log-likelihood:", round(modelo.llf, 1), "  modello nullo:", round(modelo.llnull, 1))
print("Pseudo R2:", round(modelo.prsquared, 3), "  AIC:", round(modelo.aic, 1))

### Lo stesso modello con `sm.GLM` (famiglia Binomiale)

Un GLM con famiglia Binomiale e collegamento logit (quello predefinito) è esattamente la regressione logistica: i coefficienti coincidono. (L'AIC compare nell'output; lo useremo per confrontare modelli nella lezione 12.)

In [ ]:
glm_bin = smf.glm("abandona ~ antiguedad + gasto + tickets + plan_anual", data=df, family=sm.families.Binomial()).fit()
print("Differenza massima tra coefficienti logit e GLM:", float(np.abs(glm_bin.params - modelo.params).max()) < 1e-8)

---
## 6. Odds ratio con intervallo di confidenza

Applichiamo `exp` ai coefficienti **e agli estremi del loro intervallo**. Riscaliamo anche a una variazione rilevante (12 mesi, 10 €).

In [ ]:
tabla_or = pd.DataFrame({"odds ratio": np.exp(modelo.params),
                         "IC 2.5%": np.exp(ic[0]), "IC 97.5%": np.exp(ic[1])})
display(tabla_or.drop(index="Intercept").round(3))

print("12 mesi in più di anzianità moltiplicano gli odds per", round(np.exp(12 * modelo.params["antiguedad"]), 3))
print("10 € in più di spesa moltiplicano gli odds per", round(np.exp(10 * modelo.params["gasto"]), 3))

### Esperimento guidato 2: odds ratio rispetto a probabilità

L'odds ratio di `tickets` (≈ 1,68) è lo stesso per qualsiasi cliente. Lo applichiamo a probabilità di partenza molto diverse e guardiamo cosa succede alla probabilità. **Prima di eseguire, prevedi:** in quale punto di partenza l'odds ratio somiglia di più al rapporto tra probabilità?

In [ ]:
or_tickets = float(np.exp(modelo.params["tickets"]))
p_partida = np.array([0.01, 0.05, 0.20, 0.50, 0.80, 0.95])
p_final = sigmoide(a_logodds(p_partida) + modelo.params["tickets"])
efecto = pd.DataFrame({"p di partenza": p_partida, "p con +1 ticket": p_final,
                       "rapporto tra probabilità": p_final / p_partida,
                       "differenza (punti)": (p_final - p_partida) * 100})
print("Odds ratio di tickets:", round(or_tickets, 3))
display(efecto.round(3))

plt.plot(p_partida * 100, (p_final - p_partida) * 100, marker="o")
plt.title("Stesso odds ratio, diversa variazione di probabilità")
plt.xlabel("probabilità di partenza (%)")
plt.ylabel("variazione di p con +1 ticket (punti)")
plt.show()

**Cosa osservi?** Con p = 0,01 il rapporto tra probabilità è quasi uguale all'odds ratio (≈ 1,67): evento raro, odds ≈ probabilità. Salendo la probabilità di partenza il rapporto scende verso 1 (con p = 0,95 appena 1,02), e la variazione in punti cresce fino a un massimo vicino a p = 0,5 e poi ridiscende. **Lo stesso odds ratio dà variazioni molto diverse**.

Lo stesso con dati reali: la tabella semplice dei clienti con **3 o più ticket** rispetto agli altri (senza modello).

In [ ]:
df["muchos_tickets"] = (df["tickets"] >= 3).astype(int)
g = df.groupby("muchos_tickets")["abandona"].agg(["count", "sum", "mean"])
g.columns = ["clienti", "abbandonano", "p"]
g["odds"] = g["p"] / (1 - g["p"])
display(g.round(3))
rr = g.loc[1, "p"] / g.loc[0, "p"]
odds_r = g.loc[1, "odds"] / g.loc[0, "odds"]
print("Rischio relativo:", round(rr, 2), " Odds ratio:", round(odds_r, 2))

**Cosa osservi?** La probabilità di abbandonare si moltiplica per ≈ 2,1 (rischio relativo), ma gli odds si moltiplicano per ≈ 3,0. Poiché l'abbandono è frequente (24 %), l'odds ratio **sovrastima** la variazione di probabilità. Dire "il triplo della probabilità" sarebbe un errore.

---
## 7. Variabili categoriali: dummy e categoria di riferimento

Raggruppiamo l'anzianità in tre fasce e lasciamo che `statsmodels` crei le variabili dummy con `C(...)`. `Treatment(reference="nuevo")` fissa la categoria di riferimento.

In [ ]:
df["tramo"] = pd.cut(df["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
display(df.groupby("tramo")["abandona"].agg(["count", "mean"]).round(3))

m_tramo = smf.logit("abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual", data=df).fit(disp=0)
ic_t = np.exp(m_tramo.conf_int())
tabla_t = pd.DataFrame({"coef": m_tramo.params, "odds ratio": np.exp(m_tramo.params), "IC 2.5%": ic_t[0], "IC 97.5%": ic_t[1]})
display(tabla_t.filter(like="tramo", axis=0).round(3))

b_vet, b_med = [m_tramo.params[f"C(tramo, Treatment(reference='nuevo'))[T.{t}]"] for t in ("veterano", "medio")]
print("Odds ratio veterano rispetto a medio (non compare nella tabella):", round(np.exp(b_vet - b_med), 3))

**Cosa osservi?** Ogni odds ratio confronta con il riferimento `nuevo`: il `veterano` ha odds di abbandono ≈ 75 % inferiori (odds ratio ≈ 0,25) e il `medio` ≈ 33 % inferiori. Per confrontare `veterano` con `medio` bisogna dividere i due odds ratio (≈ 0,38). Se cambiassimo il riferimento, i coefficienti cambierebbero valore: per questo bisogna dire sempre "rispetto a cosa".

---
## 8. Probabilità previste ed effetto marginale medio

L'odds ratio di `tickets` è lo stesso per tutti i clienti, ma **la variazione di probabilità dipende dal punto di partenza**. Confrontiamo due profili e poi facciamo la media su tutti i clienti (effetto marginale medio).

In [ ]:
perfiles = pd.DataFrame({
    "antiguedad": [48, 48, 6, 6], "gasto": [30, 30, 50, 50],
    "tickets": [0, 1, 3, 4], "plan_anual": [1, 1, 0, 0]},
    index=["tranquillo, 0 ticket", "tranquillo, 1 ticket",
           "a rischio, 3 ticket", "a rischio, 4 ticket"])
perfiles["p"] = modelo.predict(perfiles)
display(perfiles.round(3))
print("Variazione di p con +1 ticket (tranquillo):", round((perfiles["p"].iloc[1] - perfiles["p"].iloc[0]) * 100, 1), "punti percentuali")
print("Variazione di p con +1 ticket (a rischio):", round((perfiles["p"].iloc[3] - perfiles["p"].iloc[2]) * 100, 1), "punti percentuali")

In [ ]:
t = np.arange(0, 9)
for nombre, a, g_, pl in [("tranquillo", 48, 30, 1), ("a rischio", 6, 50, 0)]:
    datos_t = pd.DataFrame({"antiguedad": a, "gasto": g_, "tickets": t, "plan_anual": pl})
    plt.plot(t, modelo.predict(datos_t), marker="o", label=nombre)
plt.title("Probabilità di abbandono in base ai ticket")
plt.xlabel("tickets")
plt.ylabel("probabilità prevista")
plt.legend()
plt.show()

meses = np.arange(1, 73)
for pl, estilo in [(0, "-"), (1, "--")]:
    datos_m = pd.DataFrame({"antiguedad": meses, "gasto": 40, "tickets": 1, "plan_anual": pl})
    plt.plot(meses, modelo.predict(datos_m), estilo, label=f"plan_anual = {pl}")
plt.title("Probabilità di abbandono in base all'anzianità e al piano")
plt.xlabel("anzianità (mesi)")
plt.ylabel("probabilità prevista")
plt.legend()
plt.show()

**Cosa osservi?** Un ticket in più sposta la probabilità di circa 2 punti nel cliente tranquillo e di circa 8 in quello a rischio, con **lo stesso odds ratio**. Le curve sono a S, non rette. Nel secondo grafico, il piano annuale ha sempre lo stesso odds ratio (≈ 0,47), ma la distanza tra le due curve è grande con poca anzianità (probabilità alte) e quasi scompare con molta (probabilità basse).

### Esperimento guidato 3: l'effetto marginale di un ticket, cliente per cliente

Calcoliamo la variazione di probabilità aggiungendo un ticket **per ciascuno dei 2 000 clienti** e guardiamo la sua distribuzione. La sua media è l'effetto marginale medio (AME). **Prima di eseguire, prevedi:** l'effetto sarà simile per tutti i clienti? Tra quali valori ti aspetti che si muova?

In [ ]:
p_antes = modelo.predict(df)
df_mas = df.copy()
df_mas["tickets"] = df_mas["tickets"] + 1
efecto_ind = (modelo.predict(df_mas) - p_antes) * 100
print("Effetto marginale medio (AME) di +1 ticket:", round(efecto_ind.mean(), 1), "punti percentuali")
print("Effetto individuale minimo e massimo:", round(efecto_ind.min(), 1), "e", round(efecto_ind.max(), 1))
me = modelo.get_margeff()
print("Versione con le derivate (get_margeff):", round(me.margeff[list(modelo.params.index[1:]).index("tickets")] * 100, 1))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.hist(efecto_ind, bins=30, edgecolor="white")
a1.axvline(efecto_ind.mean(), color="red", linestyle="--", label="AME")
a1.set_title("Effetto di +1 ticket, cliente per cliente")
a1.set_xlabel("variazione di p (punti)"); a1.legend()
a2.scatter(p_antes * 100, efecto_ind, s=6, alpha=0.4)
a2.set_title("Effetto in base alla probabilità di partenza")
a2.set_xlabel("p di partenza (%)"); a2.set_ylabel("variazione di p (punti)")
plt.show()

**Cosa osservi?** L'effetto **non è lo stesso per tutti**: va da ≈ 1,6 a ≈ 12,9 punti, e l'AME (≈ 9,3) è un riassunto. Il diagramma di dispersione mostra la relazione con la probabilità di partenza: l'effetto è piccolo agli estremi e massimo nella zona centrale. Un unico "effetto di un ticket" in punti ha senso solo come media.

---
## 9. Separazione perfetta (solo un avvertimento)

Se una variabile separa completamente gli 0 dagli 1, la verosimiglianza non ha un massimo finito e i coefficienti esplodono. A seconda della versione di `statsmodels`, vedrai un avviso o un errore di matrice singolare. **Non si interpreta.**

In [ ]:
d_sep = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6, 7, 8], "y": [0, 0, 0, 0, 1, 1, 1, 1]})   # x separa perfettamente y
hubo_problema = False
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    try:
        smf.logit("y ~ x", data=d_sep).fit(disp=0)
    except Exception:
        hubo_problema = True
hubo_problema = hubo_problema or len(avisos) > 0
print("statsmodels ha avvisato o è fallito davanti alla separazione perfetta?", hubo_problema)

---
## 10. La famiglia GLM e la regressione di Poisson (solo come idea)

Un GLM ha una **componente casuale**, un **predittore lineare** e una **funzione di collegamento**. Cambia solo la famiglia in `smf.glm`.

In [ ]:
familias = pd.DataFrame({
    "componente casuale": ["Normale (Gaussian)", "Binomial", "Poisson"],
    "collegamento": ["identità", "logit", "logaritmo"],
    "risposta tipica": ["continua (prezzo)", "0/1 (abandona)", "conteggio (ticket)"]},
    index=["lineare", "logistica", "Poisson"])
display(familias)

# Poisson: il numero di ticket è un conteggio. exp(coef) moltiplica il TASSO (conteggio medio), non gli odds
poisson = smf.glm("tickets ~ gasto", data=df, family=sm.families.Poisson()).fit()
print("Coefficiente di gasto (log del tasso per €):", round(poisson.params["gasto"], 4))
print("10 € in più di spesa moltiplicano il n. medio di ticket per", round(np.exp(10 * poisson.params["gasto"]), 3))

**Cosa osservi?** Il codice è lo stesso della logistica: cambia solo `family`. Il coefficiente si interpreta sulla scala del collegamento (qui, log del tasso) e `exp(β)` moltiplica il **tasso medio di ticket**, non degli odds. Poisson qui è solo un esempio del perché esistono i GLM: per i conteggi la lineare e la logistica non servono.

---
## Riepilogo della lezione

- La regressione lineare non serve per uno 0/1; il modello logistico modella i **log-odds** come funzione lineare e restituisce probabilità in (0, 1).
- I coefficienti sono **log-odds**; `exp(β)` è l'**odds ratio** (non la probabilità né il rischio relativo). Si riporta con il suo intervallo.
- Con variabili categoriali, ogni odds ratio confronta con la **categoria di riferimento**.
- La variazione di probabilità dipende dal punto di partenza; l'**effetto marginale medio** la riassume.
- Una separazione perfetta produce coefficienti che non si interpretano.
- I **GLM** unificano lineare, logistica e Poisson: componente casuale + predittore lineare + collegamento.

**Prossimo passo:** il notebook di pratica di questa lezione e poi la lezione 12: Multicollinearità, selezione delle variabili e validazione.